# Kalshi BTC 15-Minute Settlement Markets
Fetch all BTC settlement markets from the `KXBTC15M` series on Kalshi for 15-minute periods.

In [2]:
import requests
import pandas as pd
from datetime import datetime

In [3]:
BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"
SERIES_TICKER = "KXBTC15M"

# Verify series exists
response = requests.get(f"{BASE_URL}/series/{SERIES_TICKER}")
series_data = response.json()
print(f"Series: {series_data['series']['title']}")
print(f"Frequency: {series_data['series']['frequency']}")
print(f"Category: {series_data['series']['category']}")

Series: Bitcoin price up down
Frequency: fifteen_min
Category: Crypto


In [4]:
# Fetch ALL settled BTC 15-minute markets with pagination
all_markets = []
cursor = None
page = 0

while True:
    params = {
        "series_ticker": SERIES_TICKER,
        "status": "settled",
        "limit": 200,
    }
    if cursor:
        params["cursor"] = cursor

    resp = requests.get(f"{BASE_URL}/markets", params=params)
    data = resp.json()

    markets = data.get("markets", [])
    all_markets.extend(markets)
    page += 1
    print(f"Page {page}: fetched {len(markets)} markets (total so far: {len(all_markets)})")

    cursor = data.get("cursor")
    if not cursor or not markets:
        break

print(f"\nTotal settled BTC 15-min markets fetched: {len(all_markets)}")

Page 1: fetched 200 markets (total so far: 200)
Page 2: fetched 200 markets (total so far: 400)
Page 3: fetched 200 markets (total so far: 600)
Page 4: fetched 200 markets (total so far: 800)
Page 5: fetched 200 markets (total so far: 1000)
Page 6: fetched 200 markets (total so far: 1200)
Page 7: fetched 200 markets (total so far: 1400)
Page 8: fetched 200 markets (total so far: 1600)
Page 9: fetched 200 markets (total so far: 1800)
Page 10: fetched 200 markets (total so far: 2000)
Page 11: fetched 200 markets (total so far: 2200)
Page 12: fetched 200 markets (total so far: 2400)
Page 13: fetched 200 markets (total so far: 2600)
Page 14: fetched 200 markets (total so far: 2800)
Page 15: fetched 200 markets (total so far: 3000)
Page 16: fetched 200 markets (total so far: 3200)
Page 17: fetched 200 markets (total so far: 3400)
Page 18: fetched 200 markets (total so far: 3600)
Page 19: fetched 200 markets (total so far: 3800)
Page 20: fetched 200 markets (total so far: 4000)
Page 21: fetc

In [8]:
# Build DataFrame from settled markets
df = pd.DataFrame([{
    "ticker": m["ticker"],
    "title": m["title"],
    "event_ticker": m["event_ticker"],
    "open_time": m.get("open_time"),
    "close_time": m.get("close_time"),
    "expiration_time": m.get("expiration_time"),
    "yes_bid": m.get("yes_bid_dollars"),
    "yes_ask": m.get("yes_ask_dollars"),
    "last_price": m.get("last_price_dollars"),
    "volume": m.get("volume_fp"),
    "result": m.get("result"),
    "status": m.get("status"),
} for m in all_markets])

if not df.empty:
    df["close_time"] = pd.to_datetime(df["close_time"])
    df["open_time"] = pd.to_datetime(df["open_time"])
    df = df.sort_values("close_time", ascending=False).reset_index(drop=True)

print(f"DataFrame shape: {df.shape}")
df.head(10)

DataFrame shape: (5800, 12)


,ticker,title,event_ticker,open_time,close_time,expiration_time,yes_bid,yes_ask,last_price,volume,result,status
0,KXBTC15M-26APR191100-00,BTC price up in next 15 mins?,KXBTC15M-26APR191100,2026-04-19 14:45:00+00:00,2026-04-19 15:00:00+00:00,2026-04-26T15:00:00Z,0.0000,1.0000,0.9990,216457.08,yes,finalized
1,KXBTC15M-26APR191045-45,BTC price up in next 15 mins?,KXBTC15M-26APR191045,2026-04-19 14:30:00+00:00,2026-04-19 14:45:00+00:00,2026-04-26T14:45:00Z,0.0000,1.0000,0.9990,266079.70,yes,finalized
2,KXBTC15M-26APR191030-30,BTC price up in next 15 mins?,KXBTC15M-26APR191030,2026-04-19 14:15:00+00:00,2026-04-19 14:30:00+00:00,2026-04-26T14:30:00Z,0.0000,1.0000,0.9990,138428.82,yes,finalized
3,KXBTC15M-26APR191015-15,BTC price up in next 15 mins?,KXBTC15M-26APR191015,2026-04-19 14:00:00+00:00,2026-04-19 14:15:00+00:00,2026-04-26T14:15:00Z,0.0000,1.0000,0.0010,151103.42,no,finalized
4,KXBTC15M-26APR191000-00,BTC price up in next 15 mins?,KXBTC15M-26APR191000,2026-04-19 13:45:00+00:00,2026-04-19 14:00:00+00:00,2026-04-26T14:00:00Z,0.0000,1.0000,0.0010,131621.39,no,finalized
5,KXBTC15M-26APR190945-45,BTC price up in next 15 mins?,KXBTC15M-26APR190945,2026-04-19 13:30:00+00:00,2026-04-19 13:45:00+00:00,2026-04-26T13:45:00Z,0.0000,1.0000,0.0060,181788.53,no,finalized
6,KXBTC15M-26APR190930-30,BTC price up in next 15 mins?,KXBTC15M-26APR190930,2026-04-19 13:15:00+00:00,2026-04-19 13:30:00+00:00,2026-04-26T13:30:00Z,0.0000,1.0000,0.9990,147023.25,yes,finalized
7,KXBTC15M-26APR190915-15,BTC price up in next 15 mins?,KXBTC15M-26APR190915,2026-04-19 13:00:00+00:00,2026-04-19 13:15:00+00:00,2026-04-26T13:15:00Z,0.0000,1.0000,0.9990,121063.73,yes,finalized
8,KXBTC15M-26APR190900-00,BTC price up in next 15 mins?,KXBTC15M-26APR190900,2026-04-19 12:45:00+00:00,2026-04-19 13:00:00+00:00,2026-04-26T13:00:00Z,0.0000,1.0000,0.9990,121789.64,yes,finalized
9,KXBTC15M-26APR190845-45,BTC price up in next 15 mins?,KXBTC15M-26APR190845,2026-04-19 12:30:00+00:00,2026-04-19 12:45:00+00:00,2026-04-26T12:45:00Z,0.0000,1.0000,0.0010,190202.33,no,finalized


In [ ]:
# Summary stats
if not df.empty:
    print(f"Date range: {df['close_time'].min()} to {df['close_time'].max()}")
    print(f"Total markets: {len(df)}")
    print(f"\nResult distribution:")
    print(df["result"].value_counts())
    print(f"\nVolume stats:")
    print(df["volume"].describe())

Date range: 2026-02-17 05:15:00+00:00 to 2026-04-19 15:00:00+00:00
Total markets: 5800

Result distribution:
result
yes    2959
no     2841
Name: count, dtype: int64

Volume stats:
count     5800
unique    5754
top       0.00
freq         9
Name: volume, dtype: object


In [1]:
print(all_markets[0])

NameError: name 'all_markets' is not defined